# Etapa 3: procesamiento distribuido con PySpark

En esta sección utilizamos la API de RDDs de Apache Spark para resolver preguntas de negocio que serían ineficientes o impracticables en el motor relacional de PostgreSQL (como el análisis léxico de textos libres y los cálculos combinatorios masivos).

In [1]:
# inicialización del entorno
from pyspark import SparkContext
import re
import itertools

# como estamos en el contenedor oficial de docker, la configuración es directa
sc = SparkContext("local[*]", "AnalisisDelivery")
print("¡Contexto de Spark inicializado correctamente!")

¡Contexto de Spark inicializado correctamente!


In [2]:
!ls -l /home/jovyan/work/

total 392
-rwxrwxrwx 1 root root  46004 Jun 19 21:53 mapreduce_spark.ipynb
-rwxrwxrwx 1 root root 336459 Jun 19 21:04 pedidos.csv
-rwxrwxrwx 1 root root  10493 Jun 19 21:03 platos.csv


## 1. Análisis léxico de ingredientes (frecuencia de palabras clave)

**Pregunta de negocio:** ¿Cuáles son los ingredientes o conceptos más recurrentes en las descripciones de los platos que ofrecen los restaurantes? (Útil para tendencias de consumo).
* **Fase de map:** Filtramos los platos válidos, normalizamos el texto de la descripción (minúsculas, sin puntuación) y separamos las palabras. Excluimos conectores gramaticales ("stopwords"). Emitimos una tupla `(palabra, 1)`.
* **Fase de reduce:** Agrupamos por la clave (palabra) y aplicamos una suma acumulativa.

In [ ]:
# cargamos el archivo de platos usando la ruta absoluta del contenedor
rdd_platos = sc.textFile("/home/jovyan/work/platos.csv")
encabezado_platos = rdd_platos.first()

def limpiar_y_tokenizar(linea):
    campos = linea.split('|')
    # aseguramos que la línea tenga la columna de descripción (índice 3)
    if len(campos) > 3:
        descripcion = campos[3]
        # removemos puntuación y pasamos a minúsculas
        texto_limpio = re.sub(r'[^\w\s]', '', descripcion.lower())
        palabras = texto_limpio.split()
        
        # lista de palabras ignoradas (stopwords)
        stopwords = {'con', 'de', 'y', 'el', 'la', 'en', 'al', 'los', 'las', 'un', 'una', 'para'}
        return [p for p in palabras if p not in stopwords and len(p) > 2]
    return []

# construcción del pipeline
ingredientes_frecuentes = rdd_platos.filter(lambda linea: linea != encabezado_platos) \
    .flatMap(limpiar_y_tokenizar) \
    .map(lambda palabra: (palabra, 1)) \
    .reduceByKey(lambda a, b: a + b) \
    .sortBy(lambda x: x[1], ascending=False)

# ejecución de la acción
print("top 10 ingredientes/palabras en descripciones:")
for ingrediente, cantidad in ingredientes_frecuentes.take(10):
    print(f"- {ingrediente}: {cantidad} menciones")

top 10 ingredientes/palabras en descripciones:
- tomate: 29 menciones
- elección: 19 menciones
- doble: 16 menciones
- palta: 15 menciones
- piezas: 14 menciones
- corte: 14 menciones
- medallones: 12 menciones
- panceta: 12 menciones
- lentejas: 12 menciones
- lechuga: 12 menciones


## 2. Filtro colaborativo: pares de restaurantes (market basket analysis)

**Pregunta de negocio:** ¿Qué restaurantes suelen compartir la misma clientela? Esto permite generar asociaciones estratégicas o recomendar a un usuario un restaurante basándonos en lo que comen personas con gustos similares.
* **Fase de map:** Obtenemos la tupla `(id_usuario, id_restaurante)` y aplicamos un filtro de unicidad.
* **Fase de reduce:** Agrupamos los restaurantes en listas asociadas a cada usuario. Luego (segundo map) calculamos las combinaciones posibles de pares dentro de esa lista, y las acumulamos mediante una suma.

In [6]:
# cargamos el archivo de pedidos usando la ruta absoluta del contenedor
rdd_pedidos = sc.textFile("/home/jovyan/work/pedidos.csv")
encabezado_pedidos = rdd_pedidos.first()

# obtenemos visitas únicas por usuario
# formato de pedidos: id_pedido|id_usuario|id_restaurante|id_repartidor|id_promocion...
visitas_unicas = rdd_pedidos.filter(lambda linea: linea != encabezado_pedidos) \
    .map(lambda linea: linea.split('|')) \
    .map(lambda campos: (int(campos[1]), int(campos[2]))) \
    .distinct()

# agrupamos el historial y calculamos combinaciones
pares_restaurantes = visitas_unicas.groupByKey() \
    .mapValues(list) \
    .filter(lambda x: len(x[1]) > 1) \
    .flatMap(lambda x: itertools.combinations(sorted(x[1]), 2)) \
    .map(lambda par: (par, 1)) \
    .reduceByKey(lambda a, b: a + b) \
    .sortBy(lambda x: x[1], ascending=False)

print("top 5 pares de restaurantes consumidos por los mismos clientes:")
for par, coincidencias in pares_restaurantes.take(5):
    print(f"restaurantes {par[0]} y {par[1]} -> {coincidencias} usuarios en común")

top 5 pares de restaurantes consumidos por los mismos clientes:
restaurantes 18 y 36 -> 19 usuarios en común
restaurantes 43 y 48 -> 19 usuarios en común
restaurantes 4 y 33 -> 19 usuarios en común
restaurantes 16 y 26 -> 18 usuarios en común
restaurantes 23 y 41 -> 17 usuarios en común


## 3. Identificación de "cazadores de promociones"

**Pregunta de negocio:** ¿Qué usuarios tienen la mayor tasa de uso de cupones de descuento respecto a su cantidad total de compras? (Condición: mínimo 3 pedidos realizados).
* **Fase de map:** Analizamos si la transacción usó un cupón. Emitimos `(id_usuario, (1, 1))` si usó promoción, o `(id_usuario, (1, 0))` si no la usó. El primer número es el contador de pedidos, el segundo es el contador de promociones.
* **Fase de reduce:** Acumulamos ambos contadores para saber el total de pedidos y promociones por cliente. Finalmente, un mapValues calcula el porcentaje y filtramos aquellos clientes esporádicos.

In [7]:
def analizar_promociones(linea):
    campos = linea.split('|')
    id_usuario = int(campos[1])
    # verificamos si la columna id_promocion (índice 4) tiene datos
    id_promocion = campos[4].strip()
    
    # si no está vacío y no dice NULL, asumimos que usó promoción
    uso_promo = 1 if (id_promocion and id_promocion.upper() != 'NULL') else 0
    
    return (id_usuario, (1, uso_promo))

tasa_promociones = rdd_pedidos.filter(lambda linea: linea != encabezado_pedidos) \
    .map(analizar_promociones) \
    .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1])) \
    .filter(lambda tupla: tupla[1][0] >= 3) \
    .mapValues(lambda totales: round((totales[1] / totales[0]) * 100, 2)) \
    .sortBy(lambda x: x[1], ascending=False)

print("top 5 usuarios cazadores de ofertas (porcentaje de pedidos con promo):")
for usuario, porcentaje in tasa_promociones.take(5):
    print(f"usuario {usuario}: usó cupones en el {porcentaje}% de sus pedidos")

top 5 usuarios cazadores de ofertas (porcentaje de pedidos con promo):
usuario 512: usó cupones en el 100.0% de sus pedidos
usuario 780: usó cupones en el 100.0% de sus pedidos
usuario 280: usó cupones en el 100.0% de sus pedidos
usuario 908: usó cupones en el 100.0% de sus pedidos
usuario 736: usó cupones en el 100.0% de sus pedidos


## Nota sobre la ejecución y lazy evaluation
En el desarrollo de este notebook hemos hecho uso intensivo del concepto de evaluación perezosa (*lazy evaluation*). Las transformaciones aplicadas sobre los RDDs (tales como `map`, `filter`, `flatMap` y `reduceByKey`) no procesan la información de manera inmediata. En su lugar, el gestor del clúster construye un grafo acíclico dirigido (DAG) con la secuencia lógica de pasos.

El cómputo distribuido real en los nodos se desencadena de manera optimizada únicamente al llegar a las acciones finales de cada bloque, que en nuestro caso corresponden a la instrucción `take()`. Esto evita saturar la memoria RAM y permite a Spark calcular la mejor estrategia de partición de datos antes de ejecutar el trabajo.